In [15]:
import pandas as pd
from io import StringIO

In [23]:
id = 1
titleType = 'hoh'
originalTitle = 'popo'
isAdult = 1
startYear = 1
runtimeMinutes = 2
genres = 'Comedy'
url = 'https://poop.com'

In [26]:
query = f"""INSERT INTO title (title_id, titleType, originalTitle, isAdult, startYear, runtimeMinutes, genres, url)
VALUES
('{id}', '{titleType}', '{originalTitle}', '{isAdult}',{startYear},{runtimeMinutes},'{genres}', '{url}');
"""

In [27]:
query

"INSERT INTO title (title_id, titleType, originalTitle, isAdult, startYear, runtimeMinutes, genres, url)\nVALUES\n('1', 'hoh', 'popo', '1',1,2,'Comedy', 'https://poop.com');\n"

In [21]:
data = """
tconst	titleType	primaryTitle	originalTitle	isAdult	startYear	endYear	runtimeMinutes	genres	img_url_asset
tt0000929	short	Klebolin klebt alles	Klebolin klebt alles	0	1990	\N	\N	Comedy,Short	\N
tt0000977	short	Mutterliebe	Mutterliebe	0	1990	\N	\N	Short	\N
"""
data_file = StringIO(data)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 172-173: malformed \N character escape (2273244764.py, line 5)

In [19]:
data = pd.read_csv(data_file,sep='\t')

In [20]:
data

,tconst titleType primaryTitle originalTitle isAdult startYear endYear runtimeMinutes genres img_url_asset
0,tt0000929 short Klebolin klebt alles Klebolin ...
1,tt0000977 short Mutterliebe Mutterliebe 0 1990...


In [2]:
titles = pd.read_csv("../../truncated_data/truncated_title.basics.tsv", sep='\t')

In [19]:
titles

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,img_url_asset
0,tt0000929,short,Klebolin klebt alles,Klebolin klebt alles,0,1990,\N,\N,"Comedy,Short",\N
1,tt0000977,short,Mutterliebe,Mutterliebe,0,1990,\N,\N,Short,\N
2,tt0034841,short,Hen Hop,Hen Hop,0,1994,\N,4,"Animation,Short",https://image.tmdb.org/t/p/{width_variable}/88...
3,tt0040844,short,Crossroads of Laredo,Crossroads of Laredo,0,1995,\N,30,"Short,Western",https://image.tmdb.org/t/p/{width_variable}/at...
4,tt0078006,short,Norman and the Killer,Norman and the Killer,0,1991,\N,30,"Horror,Short",\N
...,...,...,...,...,...,...,...,...,...,...
495,tt0103054,tvEpisode,Wer zweimal stirbt,Wer zweimal stirbt,0,1991,\N,88,"Crime,Drama,Mystery",\N
496,tt0103062,tvEpisode,Tell Me That You Love Me,Tell Me That You Love Me,0,1991,\N,98,"Comedy,Crime,Drama",https://image.tmdb.org/t/p/{width_variable}/hE...
497,tt0103071,tvEpisode,Thanners neuer Job,Thanners neuer Job,0,1991,\N,83,"Crime,Drama,Mystery",\N
498,tt0103145,tvEpisode,Two Teens and a Baby,Two Teens and a Baby,0,1992,\N,\N,"Adventure,Comedy,Drama",\N


In [15]:
episode = pd.read_csv("../../truncated_data/truncated_title.episode.tsv", sep='\t')

In [14]:
genres = []
for index,row in titles.iterrows():
    for item in row['genres'].split(','):
        genres.append(item)

In [16]:
set(genres)

{'Action',
 'Adult',
 'Adventure',
 'Animation',
 'Biography',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'History',
 'Horror',
 'Music',
 'Musical',
 'Mystery',
 'News',
 'Romance',
 'Sci-Fi',
 'Short',
 'Sport',
 'Thriller',
 'War',
 'Western',
 '\\N'}

In [18]:
with open('../../../SQL/population/done/title_insertions.sql','w') as f:
    for index, row in titles.iterrows():
        id = row['tconst'] if row['tconst'] != r"\N" else "NULL"
        titleType = row['titleType'] if row['titleType'] != r"\N" else "NULL"
        titleType = titleType.replace("'", '"')
        primaryTitle = row['primaryTitle'] if row['primaryTitle'] != r"\N" else "NULL"
        primaryTitle = primaryTitle.replace("'", '"')
        originalTitle = row['originalTitle'] if row['originalTitle'] != r"\N" else "NULL"
        originalTitle = originalTitle.replace("'", '"')
        isAdult = row['isAdult'] if row['isAdult'] != r"\N" else "NULL"
        startYear = row['startYear'] if row['startYear'] != r"\N" else "NULL"
        endYear = row['endYear'] if row['endYear'] != r"\N" else "NULL"
        runtimeMinutes = row['runtimeMinutes'] if row['runtimeMinutes'] != r"\N" else "NULL"
        genres = row['genres'] if row['genres'] != r"\N" else "NULL"
        url = row['img_url_asset'] if row['img_url_asset'] != r"\N" else "NULL"
        url = url.replace("{width_variable}", "original")
        f.write('INSERT INTO title (title_id, titleType, originalTitle, isAdult, startYear, runtimeMinutes, genres, url)\n')
        f.write('VALUES ')
        f.write(f"('{id}', '{titleType}', '{originalTitle}', '{isAdult}',{startYear},{runtimeMinutes},'{genres}', '{url}');\n")
    for item in set(list(episode['parentTconst'])):
        id = item if item != r"\N" else "NULL"
        titleType = "Series"
        primaryTitle = "NULL"
        originalTitle = "NULL"
        isAdult = 0
        startYear = "NULL"
        endYear = "NULL"
        runtimeMinutes = "NULL"
        genres = []
        df_episodes = episode[episode['parentTconst']==item]
        for index,row in df_episodes.iterrows():
            for jindex,jow in titles[titles['tconst']==row['tconst']].iterrows():
                genres.append(jow['genres'])
                if jow['isAdult'] == 1:
                    isAdult = 1
            genre_string = ""
            for item in set(genres):
                genre_string += item + ","
            genre_string = genre_string[:-1]
            genre_string = genre_string if genre_string != r"\N" else "NULL"
            f.write('INSERT INTO title (title_id, titleType, originalTitle, isAdult, startYear, runtimeMinutes, genres, url)\n')
            f.write('VALUES ')
            f.write(f"('{id}', '{titleType}', {originalTitle}, '{isAdult}',{startYear},{runtimeMinutes},'{genre_string}', NULL);\n")